# MNIST 分类 — 今日知识点练习题

请将下方 `____` 处补充完整，运行通过即为正确。

---
## 第 1 题：导入所需库

In [ ]:
# 导入 numpy
____ numpy ____ np

# 导入 PyTorch 相关库
____ torch
____ torch.nn ____ nn
____ torch.optim ____ optim
from torch.utils.data ____ DataLoader, TensorDataset

# 导入进度条和绘图工具
from tqdm ____ tqdm
____ matplotlib.pyplot ____ plt

---
## 第 2 题：读取 IDX 图像文件

In [ ]:
import struct

def read_idx_images(filepath):
    with ____(filepath, 'rb') __ f:
        # 大端序读取 4 个 int：magic, num, rows, cols
        magic, num_images, rows, cols = struct.____('>IIII', f.read(16))
        ____ magic == 2051, f"魔术数字错误: {magic}"
        
        images = np.____((num_images, rows, cols), dtype=np.uint8)
        
        ____ i ____ tqdm(____(num_images), desc="加载进度"):
            raw_data = f.____(rows * cols)
            images[i] = np.frombuffer(raw_data, dtype=np.uint8).____(rows, cols)
    
    return images

---
## 第 3 题：读取 IDX 标签文件

In [ ]:
def read_idx_labels(filepath):
    with open(filepath, 'rb') as f:
        magic, num_labels = struct.unpack('____', f.read(8))
        assert magic == ____, f"魔术数字错误: {magic}"
        
        labels = np.zeros(num_labels, dtype=np.uint8)
        for i in tqdm(range(num_labels)):
            labels[i] = struct.unpack('____', f.read(1))[0]
    
    return labels

---
## 第 4 题：数据预处理

In [ ]:
def preprocess_data(images, labels):
    # 归一化到 [0,1]
    images = images.astype(np.float32) ____ 255.0
    # 展平为 (N, 784)
    images = images.reshape(-1, ____)
    # 标签转为 int64
    labels = labels.astype(____)
    return torch.tensor(images), torch.tensor(labels)

---
## 第 5 题：创建 DataLoader

In [ ]:
batch_size = ____

train_loader = DataLoader(
    TensorDataset(____, ____),
    batch_size=batch_size,
    shuffle=____    # 训练集打乱
)

test_loader = DataLoader(
    TensorDataset(____, ____),
    batch_size=batch_size,
    shuffle=____    # 测试集不打乱
)

---
## 第 6 题：定义三层全连接神经网络

In [ ]:
class ThreeLayerFCNet(____):
    def ____(self, input_dim=784, hidden1=512, hidden2=256, num_classes=10):
        ____.____()
        self.fc1   = nn.Linear(____, ____)
        self.relu1 = nn.____()
        self.fc2   = nn.Linear(____, ____)
        self.relu2 = nn.ReLU()
        self.fc3   = nn.____(hidden2, num_classes)

    def forward(self, x):
        x = ____(self.fc1(x))   # ReLU 激活
        x = self.relu2(____)    # 第二层
        x = self.fc3(____)      # 输出层（不加 Softmax）
        return x

---
## 第 7 题：初始化模型、损失函数、优化器

In [ ]:
device = torch.device(____ if torch.cuda.is_available() ____ "cpu")
model = ThreeLayerFCNet().____(device)

# 分类任务用交叉熵损失
criterion = ____.CrossEntropyLoss()

# Adam 优化器，学习率 0.001
optimizer = optim.____(model.parameters(), lr=____)

---
## 第 8 题：训练一个 Epoch

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, epoch):
    model.____()  # 切换到训练模式
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}")
    for inputs, targets ____ pbar:
        inputs, targets = inputs.to(____), targets.to(device)
        
        # 梯度清零
        optimizer.____()
        # 前向传播
        outputs = ____(inputs)
        loss = ____(outputs, targets)
        # 反向传播
        loss.____()
        # 更新参数
        optimizer.____()
        
        running_loss += loss.____() * inputs.size(0)
        _, predicted = outputs.____(1)  # 取最大概率类别
        total += targets.size(0)
        correct += predicted.eq(____).sum().item()
    
    return running_loss / len(loader.dataset), 100.0 * correct / total

---
## 第 9 题：评估模型

In [ ]:
def evaluate(model, loader, criterion):
    model.____()  # 切换到评估模式
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.____():  # 不计算梯度
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = ____(inputs)
            loss = ____(outputs, targets)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            ____ += predicted.eq(targets).sum().item()
    
    return running_loss / len(____), 100.0 * correct / total

---
## 第 10 题：训练循环 + 自动保存最优模型

In [ ]:
num_epochs = 10
best_acc = ____  # 初始最优准确率

for epoch in range(1, ____ + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, epoch)
    test_loss, test_acc = ____(model, test_loader, criterion)
    
    # 测试准确率提升时自动保存
    if test_acc ____ best_acc:
        best_acc = test_acc
        torch.____({
            'epoch': epoch,
            'model_state_dict': model.____(),
            'test_acc': best_acc,
        }, 'checkpoints/best_model.pth')
        print(f"⭐ Epoch {epoch}: 最优模型已保存，Acc: {best_acc:.2f}%")
    else:
        print(f"   Epoch {epoch}: test Acc: {test_acc:.2f}%")

---
## 第 11 题：加载保存的模型进行预测

In [ ]:
# 加载 checkpoint
checkpoint = torch.____('checkpoints/best_model.pth', map_location=device)

# 重新实例化模型并加载参数
loaded_model = ThreeLayerFCNet().____(device)
loaded_model.____(checkpoint['model_state_dict'])
loaded_model.____()  # 切换到评估模式

# 对测试集进行预测
with torch.no_grad():
    outputs = loaded_model(____.to(device))
    _, predictions = outputs.____(1)

# 计算准确率
accuracy = (predictions.cpu() ____ y_test).sum().item() / len(y_test) * 100
print(f"加载模型准确率: {accuracy:.2f}%")

---
## 第 12 题：改用 MSELoss（需要 one-hot 编码）

In [ ]:
import torch.nn.functional as F

# one-hot 编码函数
def to_one_hot(labels, num_classes=10):
    return F.one_hot(labels, num_classes=num_classes).____()  # 转为 float

# 修改网络：末尾加 Softmax
class ThreeLayerFCNet_MSE(nn.Module):
    def __init__(self):
        super().__init__()
        # ... 相同网络结构 ...
        self.fc1 = nn.Linear(784, 512)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(512, 256)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(256, 10)
        self.softmax = nn.____(dim=1)  # 新增 Softmax

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        x = ____(x)  # 输出概率
        return x

# 使用 MSELoss
criterion_mse = nn.____()

# 训练时标签需要转为 one-hot
# targets_onehot = to_one_hot(targets)

---
## 参考答案（验证后用）

完成所有题目后，可对照 `0629.ipynb` 中的完整代码检查。